In [ ]:
# Input: MD trajectories (structure.gro + traj.xtc) for multiple replicates
# Output: Pickle files (contacts_lip.pkl, contacts_prot.pkl, relinters.pkl) containing protein-DCA and lipid-DCA contact information and distance histograms
import MDAnalysis as mda
from MDAnalysis.analysis import distances
from MDAnalysis.analysis import rdf

import numpy as np
import matplotlib.pyplot as plt
import math

In [ ]:
ldist_arr_list = []
pdist_arr_list = []
tops = ['structure_0.gro', 'structure_1.gro', 'structure_2.gro']
trjs = ['traj_0.xtc', 'traj_1.xtc', 'traj_2.xtc']
for i in range(3):
    u = mda.Universe(tops[i],
                     trjs[i])

    dca = u.select_atoms('resname DCA and not name H*')
    lips = u.select_atoms('not name H* and resname POPE or not name H* and resname POPG')
    prots = u.select_atoms('not name H* and protein')

    for ts in u.trajectory:
        ldist_arr = distances.distance_array(dca.positions,
                                            lips.positions,
                                            box=u.dimensions)
        pdist_arr = distances.distance_array(dca.positions,
                                            prots.positions,
                                            box=u.dimensions)
        ldist_arr_list.append(ldist_arr)
        pdist_arr_list.append(pdist_arr)
ldist_arr = np.array(ldist_arr_list)
pdist_arr = np.array(pdist_arr_list)

In [ ]:
all_dists = []
for timestep in ldist_arr:
    for dca_dists in timestep:
        min_dist = min(dca_dists)
        all_dists.append(min_dist)
bins = np.linspace(0.,15,125)
del_bins = bins[1] - bins[0]
counts, bin_edges = np.histogram(all_dists, bins=bins)

norm_counts = []
for i, final_edge in enumerate(bin_edges[1:]):
    r = (final_edge + bin_edges[i]) / 2
    del_r = del_bins
    vol = (4 / 3) * math.pi * ((r + del_r)**3 - (r**3))
    norm_counts.append(counts[i] / vol)

import pickle
with open('contacts_lip.pkl', 'wb') as handle:
    pickle.dump((norm_counts, bins), handle, protocol=pickle.HIGHEST_PROTOCOL)

plt.stairs(norm_counts, bins, fill=True)
plt.xlabel('distance between dca heavy atom and lipid heavy atom')
plt.show()
plt.close()

all_dists = []
for timestep in pdist_arr:
    for dca_dists in timestep:
        min_dist = min(dca_dists)
        all_dists.append(min_dist)
bins = np.linspace(0.,15,125)
del_bins = bins[1] - bins[0]
counts, bin_edges = np.histogram(all_dists, bins=bins)

norm_counts = []
for i, final_edge in enumerate(bin_edges[1:]):
    r = (final_edge + bin_edges[i]) / 2
    del_r = del_bins
    vol = (4 / 3) * math.pi * ((r + del_r)**3 - (r**3))
    norm_counts.append(counts[i] / vol)

import pickle
with open('contacts_prot.pkl', 'wb') as handle:
    pickle.dump((norm_counts, bins), handle, protocol=pickle.HIGHEST_PROTOCOL)
plt.stairs(norm_counts, bins, fill=True)
plt.xlabel('distance between dca heavy atom and protein heavy atom')
plt.show()
plt.close()

In [ ]:
cutoff = 6

lipd = ldist_arr.transpose(1, 0, 2)
prod = pdist_arr.transpose(1, 0, 2)

ts = lipd[1].shape[0]

lip_dcas = []
prot_dcas = []

for dca in lipd:
    numinter = 0
    for ts in dca:
        if min(ts) < 6:
            numinter += 1
    lip_dcas.append(numinter)
    
for dca in prod:
    numinter = 0
    for ts in dca:
        if min(ts) < 4.5:
            numinter += 1
    prot_dcas.append(numinter)


ts = lipd[1].shape[0]
lip_dcas = np.array(lip_dcas) / ts
prot_dcas = np.array(prot_dcas) / ts

rel_inters = np.array(prot_dcas) - np.array(lip_dcas)

inter_dict = {}
for i, atom in enumerate(u.select_atoms('resname DCA and not name H*')):
    inter_dict.update({atom: rel_inters[i]})
import pickle
if_data = (lip_dcas, prot_dcas, rel_inters)
with open('relinters.pkl', 'wb') as handle:
    pickle.dump(if_data, handle, protocol=pickle.HIGHEST_PROTOCOL)